In [ ]:
import h5py
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
import random

H5_PATH = '/data/users/lly/projects/PCa-HSD-LSDT/dataset/patients_dataset_postmask.h5'
f = h5py.File(H5_PATH, 'r')
patient_ids = sorted(f.keys())
print(f'Total patients: {len(patient_ids)}')
print(f'Keys in first patient: {list(f[patient_ids[0]].keys())}')

In [ ]:
# Count non-zero pixels for reference masks and post_train_mask for all patients
ref_vols = []
post_vols = []
for pid in patient_ids:
    grp = f[pid]
    ref = grp['T2_res_sam3_text_mask_0.6'][:]
    post = grp['post_train_mask'][:]
    ref_vols.append(ref.sum())
    post_vols.append(post.sum())
print(f'ref_mask sum (all patients): min={min(ref_vols)}, max={max(ref_vols)}, nonzero patients={sum(1 for v in ref_vols if v>0)}')
print(f'post_mask sum (all patients): min={min(post_vols)}, max={max(post_vols)}, nonzero patients={sum(1 for v in post_vols if v>0)}')

In [ ]:
def dice(a, b):
    a = a.astype(bool); b = b.astype(bool)
    inter = (a & b).sum()
    return 2*inter / (a.sum() + b.sum() + 1e-8)

def visualize_patient(patient_idx, slice_idx=8, cmap_alpha='jet'):
    pid = patient_ids[patient_idx]
    grp = f[pid]
    t2 = grp['T2'][:]
    adc = grp['ADC'][:]
    dwi = grp['DWI'][:]
    ref = grp['T2_res_sam3_text_mask_0.6'][:].astype(np.uint8)
    post = grp['post_train_mask'][:].astype(np.uint8)
    label = grp.attrs.get('label', '?')

    s = slice_idx
    t2_s = t2[s, ..., 0] if t2.ndim == 4 else t2[s]
    adc_s = adc[s, ..., 0] if adc.ndim == 4 else adc[s]
    dwi_s = dwi[s, ..., 0] if dwi.ndim == 4 else dwi[s]
    ref_s = ref[s, ..., 0] if ref.ndim == 4 else ref[s]
    post_s = post[s, ..., 0] if post.ndim == 4 else post[s]
    d = dice(ref_s, post_s)

    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    norm = Normalize(vmin=0, vmax=1)
    
    axes[0,0].imshow(t2_s, cmap='gray'); axes[0,0].set_title(f'Patient {pid} Slice {s} — T2 (label={label})')
    axes[0,1].imshow(adc_s, cmap='gray'); axes[0,1].set_title('ADC')
    axes[0,2].imshow(dwi_s, cmap='gray'); axes[0,2].set_title('DWI')

    axes[1,0].imshow(t2_s, cmap='gray')
    if ref_s.max() > 0:
        axes[1,0].imshow(ref_s, cmap='Reds', alpha=ref_s * 0.5, norm=norm)
    axes[1,0].set_title('T2 + ref_mask (red)')

    axes[1,1].imshow(t2_s, cmap='gray')
    if post_s.max() > 0:
        axes[1,1].imshow(post_s, cmap='Greens', alpha=post_s * 0.5, norm=norm)
    axes[1,1].set_title('T2 + post_mask (green)')

    axes[1,2].imshow(t2_s, cmap='gray')
    if ref_s.max() > 0:
        axes[1,2].imshow(ref_s, cmap='Reds', alpha=ref_s*0.4, norm=norm)
    if post_s.max() > 0:
        axes[1,2].imshow(post_s, cmap='Greens', alpha=post_s*0.4, norm=norm)
    axes[1,2].set_title(f'Overlay (ref=red, post=green)\nDice={d:.4f}')

    for ax in axes.ravel():
        ax.axis('off')
    plt.tight_layout()
    plt.show()

visualize_patient(patient_idx=7, slice_idx=8)

In [ ]:
# Random check: 5 patients, middle slice
random.seed(42)
samples = random.sample([i for i in range(100)], 6)
for idx in samples:
    visualize_patient(idx, slice_idx=4)

In [ ]:
f.close()